# 동국대 2026학년도 정시 환산점수 계산기

- 목적: 모집요강의 수능 환산점수 산출 규칙을 Python으로 구현하고, 모집요강 산출 예시와 결과가 일치하는지 검증
- 범위: 동국대 자연계열 유형①(수학과·통계학과·산업시스템공학과), 유형③(물리학과·전자전기공학부 등)
- 출처: 동국대학교 2026학년도 정시모집 요강 수능 성적 반영방법 및 산출 예시

In [2]:
import copy
import pandas as pd

## 1. 산출 규칙 정리

- 반영비율, 영어 등급별 환산표준점수, 한국사 등급별 감점, 가산비율을 딕셔너리 하나에 정리
- 계산 코드에 숫자를 직접 쓰지 않고 이 딕셔너리에서 꺼내 쓰도록 구성 → 규칙이 바뀌면 이 셀만 수정
- 입력 후 모집요강 원문과 숫자 대조 완료

In [3]:
동국대 = {
    "반영비율": {
        "국어": 0.25,
        "수학": 0.35,
        "탐구": 0.25,
        "영어": 0.15,
    },
    "영어_등급별_점수": {
        1: 200,
        2: 199,
        3: 197,
        4: 190,
        5: 180,
        6: 140,
        7: 90,
        8: 20,
        9: 0,
    },
    "한국사_등급별_감점": {
        1: 0,
        2: 0,
        3: 0,
        4: 0,
        5: -2,
        6: -4,
        7: -6,
        8: -8,
        9: -10,
    },
    "가산비율": 1.03,
}

print(동국대)

{'반영비율': {'국어': 0.25, '수학': 0.35, '탐구': 0.25, '영어': 0.15}, '영어_등급별_점수': {1: 200, 2: 199, 3: 197, 4: 190, 5: 180, 6: 140, 7: 90, 8: 20, 9: 0}, '한국사_등급별_감점': {1: 0, 2: 0, 3: 0, 4: 0, 5: -2, 6: -4, 7: -6, 8: -8, 9: -10}, '가산비율': 1.03}


## 2. 예시 학생 성적

- 모집요강 산출 예시에 나온 학생 성적을 그대로 입력
- 탐구 변환표준점수는 수능 성적 발표 후 대학이 별도 공지하는 값으로, 모집요강 예시값(67.11, 65.71) 사용

In [4]:
학생 = {
    "국어": {"표준점수": 128},
    "수학": {"표준점수": 131, "선택과목": "미적분"},
    "탐구1": {"과목": "과탐", "변환표준점수": 67.11},
    "탐구2": {"과목": "과탐", "변환표준점수": 65.71},
    "영어": {"등급": 2},
    "한국사": {"등급": 5},
}

## 3. 영역별 점수 계산 (유형③ 기준)

- 국어·수학·탐구·영어: (점수 ÷ 200) × 1,000 × 반영비율
- 수학: 선택과목이 미적분·기하인 경우에만 표준점수 3% 가산
- 탐구: 과학탐구 과목만 변환표준점수 3% 가산
- 한국사: 등급별 감점을 그대로 반영

In [5]:
# 국어 점수
국어점수 = (
    학생["국어"]["표준점수"] / 200
) * 1000 * 동국대["반영비율"]["국어"]

print(국어점수)

160.0


In [6]:
# 수학 점수
선택과목 = 학생["수학"]["선택과목"]
수학표준점수 = 학생["수학"]["표준점수"]

if 선택과목 in ["미적분", "기하"]:
    수학표준점수 *= 동국대["가산비율"]

수학점수 = (
    수학표준점수 / 200
) * 1000 * 동국대["반영비율"]["수학"]

print(수학점수)

236.12750000000003


In [7]:
# 탐구 점수
탐구합계 = 0

for 번호 in [1, 2]:
    탐구 = 학생[f"탐구{번호}"]
    점수 = 탐구["변환표준점수"]

    if 탐구["과목"] == "과탐":
        점수 *= 동국대["가산비율"]

    탐구합계 += 점수

탐구점수 = (
    탐구합계 / 200
) * 1000 * 동국대["반영비율"]["탐구"]

# 영어 점수
영어등급 = 학생["영어"]["등급"]
영어환산점수 = 동국대["영어_등급별_점수"][영어등급]
영어점수 = (
    영어환산점수 / 200
) * 1000 * 동국대["반영비율"]["영어"]

# 한국사 점수
한국사등급 = 학생["한국사"]["등급"]
한국사점수 = 동국대["한국사_등급별_감점"][한국사등급]

영역점수 = {
    "국어": 국어점수,
    "수학": 수학점수,
    "탐구": 탐구점수,
    "영어": 영어점수,
    "한국사": 한국사점수,
}

for 영역, 점수 in 영역점수.items():
    print(f"{영역}: {점수:.3f}")

합계 = round(sum(영역점수.values()), 3)
print(f"합계: {합계:.3f}")

국어: 160.000
수학: 236.128
탐구: 171.006
영어: 149.250
한국사: -2.000
합계: 714.383


- 영역별 점수는 모집요강 예시(160.000 / 236.128 / 171.006 / 149.250 / -2)와 모두 일치
- 합계는 714.383으로, 모집요강 정답 714.384와 0.001 차이 발생
- 식은 맞는데 합계만 다르므로 반올림 시점을 원인으로 의심

## 4. 반올림 시점 비교

- A: 영역별 원래 점수를 합한 뒤 합계만 소수 셋째 자리로 반올림
- B: 영역별 점수를 먼저 소수 셋째 자리로 반올림한 뒤 합산

In [8]:
점수목록 = [국어점수, 수학점수, 탐구점수, 영어점수, 한국사점수]

# A: 원래 점수를 합한 뒤 합계만 반올림
합계_A = round(sum(점수목록), 3)

# B: 각 영역을 먼저 반올림한 뒤 합산
반올림된_점수 = [round(점수, 3) for 점수 in 점수목록]
합계_B = round(sum(반올림된_점수), 3)

print("A:", 합계_A)
print("B:", 합계_B)

A: 714.383
B: 714.384


| 영역 | 원래 값 | 영역별 반올림 | 차이 |
|---|---|---|---|
| 수학 | 236.1275 | 236.128 | +0.0005 |
| 탐구 | 171.00575 | 171.006 | +0.00025 |

- A는 원래 합계 714.38325를 반올림해 714.383
- B는 수학·탐구에서 생긴 올림이 누적되어 714.384
- 모집요강 규칙 문구는 "환산점수 산출 시 소수점 넷째자리에서 반올림하여 적용"(p.37)으로, 자리수는 명시되어 있지만 영역별로 할지 합계에만 할지는 적혀 있지 않음
- 모집요강 예시 값과 일치하는 것은 B → 동국대는 영역별로 먼저 반올림한 뒤 합산하는 방식으로 판단
- 정시는 환산점수 순으로 선발하므로 합격선 근처에서는 0.001점 차이도 결과를 바꿀 수 있음 → 식뿐 아니라 계산 순서까지 원문과 맞춰야 함

## 5. 학과 유형별 계산 함수

- 유형①과 유형③의 차이는 과학탐구 가산 여부 하나 (수학 가산은 공통)
- 유형 차이를 코드의 조건문이 아니라 규칙 딕셔너리의 `탐구_과탐가산` 값으로 분리
- 합계는 B 방식(영역별 반올림 후 합산) 적용

In [9]:
# 유형별 규칙: 수학 가산은 공통, 과탐 가산 여부만 다름
DG1_규칙 = {
    **동국대,
    "유형": "DG1",
    "탐구_과탐가산": False,
}

DG3_규칙 = {
    **동국대,
    "유형": "DG3",
    "탐구_과탐가산": True,
}


def 환산점수계산(학생, 규칙):
    영역점수 = {}

    # 국어
    영역점수["국어"] = (
        학생["국어"]["표준점수"] / 200
    ) * 1000 * 규칙["반영비율"]["국어"]

    # 수학
    수학점수 = 학생["수학"]["표준점수"]
    if 학생["수학"]["선택과목"] in ["미적분", "기하"]:
        수학점수 *= 규칙["가산비율"]

    영역점수["수학"] = (
        수학점수 / 200
    ) * 1000 * 규칙["반영비율"]["수학"]

    # 탐구: 과탐 가산 규칙이 켜진 경우 해당 과목만 가산
    탐구합계 = 0
    for 번호 in [1, 2]:
        탐구 = 학생[f"탐구{번호}"]
        점수 = 탐구["변환표준점수"]

        if 규칙["탐구_과탐가산"] and 탐구["과목"] == "과탐":
            점수 *= 규칙["가산비율"]

        탐구합계 += 점수

    영역점수["탐구"] = (
        탐구합계 / 200
    ) * 1000 * 규칙["반영비율"]["탐구"]

    # 영어
    영어등급 = 학생["영어"]["등급"]
    영어점수 = 규칙["영어_등급별_점수"][영어등급]
    영역점수["영어"] = (
        영어점수 / 200
    ) * 1000 * 규칙["반영비율"]["영어"]

    # 한국사 감점
    한국사등급 = 학생["한국사"]["등급"]
    영역점수["한국사"] = 규칙["한국사_등급별_감점"][한국사등급]

    # B 방식: 영역별로 먼저 반올림한 뒤 합산
    반올림점수 = {
        영역: round(점수, 3)
        for 영역, 점수 in 영역점수.items()
    }
    반올림점수["합계"] = round(sum(반올림점수.values()), 3)

    return 반올림점수

## 6. 모집요강 예시 검증

- 모집요강에 나온 두 유형의 산출 예시를 함수 결과와 자동 비교
- 값이 다르면 에러가 나도록 `assert` 사용

In [10]:
검증목록 = [
    ("유형③", DG3_규칙, 714.384),
    ("유형①", DG1_규칙, 709.403),
]

for 이름, 규칙, 모집요강값 in 검증목록:
    결과 = 환산점수계산(학생, 규칙)["합계"]
    assert 결과 == 모집요강값, f"{이름} 불일치: {결과} != {모집요강값}"
    print(f"{이름}: {결과} (모집요강 {모집요강값}) 일치")

유형③: 714.384 (모집요강 714.384) 일치
유형①: 709.403 (모집요강 709.403) 일치


## 7. 가상 학생 테스트

| 학생 | 특징 | 확인하려는 것 |
|---|---|---|
| 학생 1 | 모집요강 예시 학생 | 공식 값 재현 |
| 학생 2 | 수학 확률과통계 | 수학 가산 제외 |
| 학생 3 | 탐구1 사탐, 탐구2 과탐 | 과탐 한 과목에만 가산 |
| 학생 4 | 영어 9등급, 한국사 4등급 | 영어 0점, 한국사 4등급 감점 없음 |

In [11]:
# 학생 1: 모집요강 예시 학생
학생1 = copy.deepcopy(학생)

# 학생 2: 선택과목만 확률과통계로 변경
학생2 = copy.deepcopy(학생)
학생2["수학"]["선택과목"] = "확률과통계"

# 학생 3: 탐구1은 사탐, 탐구2는 과탐
학생3 = copy.deepcopy(학생)
학생3["탐구1"]["과목"] = "사탐"

# 학생 4: 영어 9등급, 한국사 4등급
학생4 = copy.deepcopy(학생)
학생4["영어"]["등급"] = 9
학생4["한국사"]["등급"] = 4

학생목록 = {
    "학생 1": 학생1,
    "학생 2": 학생2,
    "학생 3": 학생3,
    "학생 4": 학생4,
}

# 학생별 DG1, DG3 합계 나란히 정리
결과표 = pd.DataFrame({
    이름: {
        "DG1": 환산점수계산(학생자료, DG1_규칙)["합계"],
        "DG3": 환산점수계산(학생자료, DG3_규칙)["합계"],
    }
    for 이름, 학생자료 in 학생목록.items()
}).T

display(결과표)

,DG1,DG3
학생 1,709.403,714.384
학생 2,702.525,707.506
학생 3,709.403,711.867
학생 4,562.153,567.134


- 학생 2: 미적분 대비 수학 점수가 236.128 → 229.25로 감소, 두 유형 모두 동일하게 반영
- 학생 3: 유형③과 유형①의 차이가 2.464점으로, 과탐 2과목인 학생 1(4.981점)보다 작음 → 가산 과목이 하나뿐이기 때문
- 학생 3의 유형① 점수가 학생 1과 같음 → 유형①은 과탐 가산이 없고 변환표준점수 값도 같게 입력했기 때문으로, 버그가 아님을 확인
- 학생 4: 영어 9등급 0점, 한국사 4등급은 감점 없음이 정상 반영

In [12]:
# 모집요강 공식 예시와 계산 결과 자동 대조
검증목록 = [
    ("유형①", DG1_규칙, 709.403),
    ("유형③", DG3_규칙, 714.384),
]

for 이름, 규칙, 모집요강값 in 검증목록:
    결과 = round(환산점수계산(학생, 규칙)["합계"], 3)
    assert 결과 == 모집요강값, f"{이름} 불일치: 계산 {결과} / 모집요강 {모집요강값}"
    print(f"{이름}: 계산 {결과} / 모집요강 {모집요강값} → 일치")

유형①: 계산 709.403 / 모집요강 709.403 → 일치
유형③: 계산 714.384 / 모집요강 714.384 → 일치


## 8. 가정과 한계

- 탐구 변환표준점수는 모집요강 예시값 사용 (실제 값은 수능 성적 발표 후 대학 공지)
- 사회탐구와 과학탐구의 변환표준점수를 같은 값으로 가정 (실제로는 다를 수 있음)
- 동국대 자연계열 유형①·③만 구현, 유형②(과탐만 가산)는 해당 학과 목록 미확인으로 제외